# Valve — normal pattern analysis on training spectrograms

Loads **normal-only** DCASE Task 2 log-mel clips for `valve` (`DCASE2020Task2LogMelDataset`), then runs `NormalPatternAnalyzer` to summarize dominant mel bands, temporal stationarity, and spectral coverage, and plots the frequency profile plus an example spectrogram with bands overlaid.

## 1. Paths and imports

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "src" / "data" / "dataset.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not find project root (src/data/dataset.py).")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020-task2-dev-dataset")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020_task2/dcase2020_task2_dev_dataset")
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "dcase2020-task2-dev-dataset"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "dataset" / "dcase2020_task2_dev_dataset"
if not DATA_PATH.exists():
    DATA_PATH = Path("../../data/dcase2020-task2-dev-dataset").resolve()

MACHINE_TYPE = "valve"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "valve"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH} (exists={DATA_PATH.exists()})")

## 2. Load normal fan spectrograms `(N, n_mels, T)`

In [ ]:
from src.data.dataset import DCASE2020Task2LogMelDataset

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset root: {DATA_PATH}")

train_ds = DCASE2020Task2LogMelDataset(root=str(DATA_PATH), machine_type=MACHINE_TYPE)
raw = train_ds.data.numpy()
normal_specs = raw[:, 0, :, :].astype(np.float64)
N, n_mels, T = normal_specs.shape
print(f"normal_specs: shape={normal_specs.shape}, dtype={normal_specs.dtype}")

## 3. `NormalPatternAnalyzer` and `visualize_analysis`

In [ ]:
class NormalPatternAnalyzer:
    """
    Analyze normal training spectrograms to discover:
    1. Dominant frequency bands
    2. Temporal stationarity
    3. Spectral coverage

    Uses ONLY normal data from training set.
    """

    def __init__(self, n_mels: int = 128, T: int = 320):
        self.n_mels = n_mels
        self.T = T

    def analyze_spectrograms(
        self,
        normal_specs: np.ndarray,  # (N, n_mels, T) - NORMAL ONLY
    ) -> dict:
        """
        Extract statistical properties from normal spectrograms.

        Returns:
            {
                'dominant_bands': list of (f_start, f_end) tuples,
                'temporal_stationarity': float in [0, 1],
                'spectral_coverage': float (fraction of bins with energy),
                'frequency_profile': (n_mels,) array of mean energy per bin
            }
        """
        N = normal_specs.shape[0]

        # 1. Frequency profile (mean energy per mel bin across time and samples)
        freq_profile = normal_specs.mean(axis=(0, 2))  # (n_mels,)

        # Normalize
        freq_profile = (freq_profile - freq_profile.min()) / (freq_profile.max() - freq_profile.min() + 1e-8)

        # 2. Find dominant frequency bands (peaks in energy)
        freq_smooth = gaussian_filter1d(freq_profile, sigma=2)
        peaks, properties = find_peaks(
            freq_smooth,
            distance=8,  # At least 8 mel bins apart
            prominence=0.15,  # Must stand out from surroundings
            width=2,  # Minimum width
        )

        # Convert peaks to bands (peak ± width/2)
        dominant_bands = []
        if len(peaks) > 0:
            widths = properties.get("widths", np.ones(len(peaks)) * 5)
            for peak, width in zip(peaks, widths):
                width_int = max(3, int(width))
                f_start = max(0, peak - width_int)
                f_end = min(self.n_mels, peak + width_int)
                dominant_bands.append((int(f_start), int(f_end)))
        else:
            # Fallback: no clear peaks, use quartile-based bands
            quartiles = np.percentile(np.arange(self.n_mels), [25, 50, 75])
            dominant_bands = [
                (0, int(quartiles[0])),
                (int(quartiles[0]), int(quartiles[1])),
                (int(quartiles[1]), int(quartiles[2])),
                (int(quartiles[2]), self.n_mels),
            ]

        # 3. Temporal stationarity (variance of frame energy over time)
        frame_energy = normal_specs.mean(axis=1)
        temporal_std = frame_energy.std(axis=1).mean()  # Absolute variation
        # Normalize by energy range instead of mean
        energy_range = frame_energy.max() - frame_energy.min()
        stationarity_raw = 1.0 - (temporal_std / (energy_range + 1e-8))
        stationarity = float(np.clip(stationarity_raw, 0, 1))

        # 4. Spectral coverage (how many mel bins have significant energy)
        threshold = freq_profile.max() * 0.1  # 10% of max
        active_bins = (freq_profile > threshold).sum()
        coverage = active_bins / self.n_mels

        return {
            "dominant_bands": dominant_bands,
            "temporal_stationarity": float(stationarity),
            "spectral_coverage": float(coverage),
            "frequency_profile": freq_profile,
            "n_dominant_bands": len(dominant_bands),
        }


def visualize_analysis(
    specs: np.ndarray,
    analysis: dict,
    machine_type: str,
    save_path: Path | None = None,
) -> None:
    """Visualize discovered patterns (for debugging/validation)."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 8))

    # Plot frequency profile with discovered bands
    axes[0].plot(analysis["frequency_profile"], label="Energy profile")
    for i, (f_start, f_end) in enumerate(analysis["dominant_bands"]):
        axes[0].axvspan(f_start, f_end, alpha=0.3, label=f"Band {i + 1}")
    axes[0].set_xlabel("Mel bin")
    axes[0].set_ylabel("Normalized energy")
    axes[0].set_title(f"{machine_type}: Frequency profile & dominant bands")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot example spectrogram with bands overlaid
    example_spec = specs[0]
    im = axes[1].imshow(example_spec, aspect="auto", origin="lower", cmap="viridis")
    for f_start, f_end in analysis["dominant_bands"]:
        axes[1].axhline(f_start, color="red", linestyle="--", alpha=0.5)
        axes[1].axhline(f_end, color="red", linestyle="--", alpha=0.5)
    axes[1].set_xlabel("Time frame")
    axes[1].set_ylabel("Mel bin")
    axes[1].set_title("Example spectrogram with bands")
    plt.colorbar(im, ax=axes[1])

    # Add text with statistics
    stats_text = (
        f"Stationarity: {analysis['temporal_stationarity']:.3f}\n"
        f"Coverage: {analysis['spectral_coverage']:.3f}\n"
        f"N bands: {analysis['n_dominant_bands']}"
    )
    axes[1].text(
        0.02,
        0.98,
        stats_text,
        transform=axes[1].transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
    )

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved figure to {save_path}")
    plt.show()
    plt.close(fig)

## 4. Run analysis and plot

In [ ]:
analyzer = NormalPatternAnalyzer(n_mels=n_mels, T=T)
analysis = analyzer.analyze_spectrograms(normal_specs)

print("dominant_bands:", analysis["dominant_bands"])
print(f"temporal_stationarity: {analysis['temporal_stationarity']:.6f}")
print(f"spectral_coverage: {analysis['spectral_coverage']:.6f}")
print(f"n_dominant_bands: {analysis['n_dominant_bands']}")

out_png = NOTEBOOK_DIR / f"normal_pattern_analysis_{MACHINE_TYPE}.png"
visualize_analysis(normal_specs, analysis, MACHINE_TYPE, save_path=out_png)